# NOTEBOOK TO GENERATE THE MASTER DATASET

This notebook is just used to fix the master dataset and compress the annotations into one column. The re-annotation, however, has to be done manually and is still subject to errors. Kindly double check any annotations that have been made in the `divergent_data_fix.csv` file. In case of any changes, the end might have to be re-run to generate the dataset again.

The `master_dataset_raw.csv` file is the original file that was provided to us WITHOUT fixing any of the data. DO NOT UNDER ANY CIRCUMSTANCE WRITE TO THIS FILE. 

The `master_dataset.csv` file will be the dataset that we will use after the annotations are flattened into one single column.

## Imports and stuff

In [45]:
import pandas as pd
import numpy as np

raw_data = pd.read_csv('master_dataset_raw.csv')

This is a simple check to see if it holds true that all rows have exactly two annotations. If the results are correct, there are only exactly two rows that have only one annotation. That is fine and we can handle that pretty easily when manually adjusting the annotations.

In [130]:
# This is a validation check to make sure that none of the rows have more than 2 columns of annotations. 
# 1 Column of annotations means we will just get the annotation and use it.

annotations = raw_data.drop('sentence_id', axis=1).drop('word_id', axis=1).drop('sentence', axis=1).drop('word', axis=1)

df = raw_data

all_valid = True

# Iterate through each row
for index, row in df.iterrows():
    # count non-NaN values in the current row
    non_nan_count = row.notna().sum()
    
    if non_nan_count != 6:
        print(f"Row {index} is invalid! It has {non_nan_count} non-NaN values.")
        all_valid = False

print(f"\nAssertion result: {all_valid}")

Row 576 is invalid! It has 5 non-NaN values.
Row 5092 is invalid! It has 5 non-NaN values.

Assertion result: False


## Flatenning the annotations
Since there are only two or less annotations per row, we will first flatten it to only two columns, disregarding the group names. These two columns will be `annotation_0` and `annotation_1`

In [ ]:
# Iterate through each row and store each instance of divergent data.

divergent_data = raw_data[['sentence_id', 'word_id', 'sentence', 'word']].copy()
divergent_data['annotation_0'] = np.nan
divergent_data['annotation_1'] = np.nan

df = raw_data.copy()

start_index = 4
end_index = 15

for index, row in df.iterrows():
    n = 0
    i = start_index

    while n < 2 and i <= end_index:
        row_data = row.iloc[i]
        if pd.isna(row.iloc[i]):
            i+=1
            continue

        dd_key = f'annotation_{n}'
        divergent_data.loc[index, dd_key] = row_data
        n += 1
        i += 1

/tmp/ipykernel_3316201/1938817676.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'FIL' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  divergent_data.loc[index, dd_key] = row_data
/tmp/ipykernel_3316201/1938817676.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'FIL' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  divergent_data.loc[index, dd_key] = row_data


This section will then generate and export a copy of the dataframe with ONLY the mismatched data for manual editing. This will be export to `divergent_data.csv` but the edited and fixed version should be (manually) saved as `divergent_data_fixed.csv`.

In [ ]:
# Export a copy of the mismatched data
# Mismatched or diverged data will be manually edited

df = divergent_data.copy()

mismatched_rows_i = list()

for index, row in df.iterrows():
    row_a = row.loc['annotation_0']
    row_b = row.loc['annotation_1']
    if row_a != row_b:
        print(f'Annotation mismatch ({index}): {row}')
        mismatched_rows_i.append(index)

print('Total count of mismatched rows: ', len(mismatched_rows_i))

df.iloc[mismatched_rows_i].to_csv('divergent_data.csv')

Annotation mismatch (89): sentence_id                                                     7
word_id                                                        89
sentence        Kung totoong may ibang tao na nagkanulo sa Diy...
word                                                        Diyos
annotation_0                                                  OTH
annotation_1                                                  FIL
Name: 89, dtype: object
Annotation mismatch (99): sentence_id                                                     7
word_id                                                        99
sentence        Kung totoong may ibang tao na nagkanulo sa Diy...
word                                                        Diyos
annotation_0                                                  OTH
annotation_1                                                  FIL
Name: 99, dtype: object
Annotation mismatch (110): sentence_id                                                     7
word_id        

## After Fixing Annotations

After fixing the divergent annotations, the below code will read the `divergent_data_fixed.csv` and put the contents of its annotations to the previously constructed dataframe with only two columns. 

Afterwhich, it will perform a final check to ensure there is no more divergent data and if so, save it into a new `master_dataset.csv`file.

In [169]:
# import the fixed divergent data
# assumes the data is all good

divergent_fixed = pd.read_csv('divergent_data_fixed.csv')

df = divergent_data.copy()

# put each of the 
for index, row in divergent_fixed.iterrows():
    ix = row.loc['indices']
    df_row = df.iloc[ix]

    if row.loc['sentence_id'] != df_row.loc['sentence_id']:
        print(f'MISMATCHED sentence_id ({ix})')
        continue
    if row.loc['word_id'] != df_row.loc['word_id']:
        print(f'MISMATCHED word_id ({ix})')
        continue
    
    df.loc[ix, 'annotation_0'] = row.loc['annotation_0']
    df.loc[ix, 'annotation_1'] = row.loc['annotation_1']

# # do final check to make sure all annotations are matching 
# # before proceeding to save the data
mismatch = False
for index, row in df.iterrows():
    row_a = row.loc['annotation_0']
    row_b = row.loc['annotation_1']
    if row_a != row_b:
        print(f'Annotation mismatch ({index}): {row}')
        mismatch = True

if not mismatch:
    fixed_df = df.drop('annotation_1', axis=1).rename(columns={'annotation_0': 'annot'})
    fixed_df.to_csv('master_dataset.csv', index=False)